In [ ]:
# ============================================================================
# Physics-Informed Neural Network for SEIR Epidemic Modeling
# Dataset: Synthetic Complete Cycle (180 days)
# Problems: Forward (prediction) and Inverse (parameter estimation)
# ============================================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================================
# Configuration
# ============================================================================

class Config:
    DATA_URL            = "https://raw.githubusercontent.com/UsamaYounas065/usama35/refs/heads/main/synthetic_complete_cycle.csv"
    TRAIN_RATIO         = 0.80
    LAYERS              = [1, 64, 64, 64, 64, 4]
    EPOCHS_FORWARD      = 15000
    EPOCHS_INVERSE      = 15000
    LEARNING_RATE       = 0.001
    PRINT_EVERY         = 500
    BETA_TRUE           = 0.45
    SIGMA_TRUE          = 0.192
    GAMMA_TRUE          = 0.077
    LAMBDA_DATA         = 10.0
    LAMBDA_PHYSICS      = 0.1
    LAMBDA_IC           = 10.0
    LAMBDA_CONSERVATION = 0.1

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {device}")

# ============================================================================
# Data Loading
# ============================================================================

print("\nLoading synthetic data...")
df         = pd.read_csv(config.DATA_URL)
population = df['population'].iloc[0]
n_days     = len(df)
n_train    = int(n_days * config.TRAIN_RATIO)
n_test     = n_days - n_train

print(f"Total days   : {n_days}")
print(f"Population   : {population:,.0f}")
print(f"Training days: {n_train}  (days 0 to {n_train-1})")
print(f"Testing days : {n_test}   (days {n_train} to {n_days-1})")

t = torch.tensor(df['time_days'].values, dtype=torch.float32).reshape(-1,1).to(device)
S = torch.tensor(df['S_norm'].values,    dtype=torch.float32).reshape(-1,1).to(device)
E = torch.tensor(df['E_norm'].values,    dtype=torch.float32).reshape(-1,1).to(device)
I = torch.tensor(df['I_norm'].values,    dtype=torch.float32).reshape(-1,1).to(device)
R = torch.tensor(df['R_norm'].values,    dtype=torch.float32).reshape(-1,1).to(device)

t_train, t_test = t[:n_train], t[n_train:]
S_train, S_test = S[:n_train], S[n_train:]
E_train, E_test = E[:n_train], E[n_train:]
I_train, I_test = I[:n_train], I[n_train:]
R_train, R_test = R[:n_train], R[n_train:]

t_train_grad = t_train.clone().requires_grad_(True)

# ============================================================================
# PINN Architecture
# ============================================================================

class PINN(nn.Module):
    def __init__(self, layers):
        super(PINN, self).__init__()
        self.net = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.net.append(nn.Linear(layers[i], layers[i+1]))
        self.activation = nn.Tanh()

    def forward(self, t):
        x = t
        for i, layer in enumerate(self.net):
            x = layer(x)
            if i < len(self.net) - 1:
                x = self.activation(x)
        return x

# ============================================================================
# Loss Functions
# ============================================================================

def derivative(y, t):
    return torch.autograd.grad(
        y, t, grad_outputs=torch.ones_like(y),
        retain_graph=True, create_graph=True)[0]

def physics_loss(t, S, E, I, R, beta, sigma, gamma):
    r1 = derivative(S, t) - (-beta * S * I)
    r2 = derivative(E, t) - (beta * S * I - sigma * E)
    r3 = derivative(I, t) - (sigma * E - gamma * I)
    r4 = derivative(R, t) - (gamma * I)
    return torch.mean(r1**2 + r2**2 + r3**2 + r4**2)

def total_loss_fn(t, model, S_t, E_t, I_t, R_t, beta, sigma, gamma):
    pred       = model(t)
    Sp, Ep, Ip, Rp = pred[:,0:1], pred[:,1:2], pred[:,2:3], pred[:,3:4]
    L_data     = torch.mean((Sp-S_t)**2 + (Ep-E_t)**2 + (Ip-I_t)**2 + (Rp-R_t)**2)
    L_phys     = physics_loss(t, Sp, Ep, Ip, Rp, beta, sigma, gamma)
    L_ic       = (Sp[0]-S_t[0])**2 + (Ep[0]-E_t[0])**2 + (Ip[0]-I_t[0])**2 + (Rp[0]-R_t[0])**2
    L_cons     = torch.mean((Sp + Ep + Ip + Rp - 1.0)**2)
    L_total    = (config.LAMBDA_DATA         * L_data  +
                  config.LAMBDA_PHYSICS      * L_phys  +
                  config.LAMBDA_IC           * L_ic    +
                  config.LAMBDA_CONSERVATION * L_cons)
    return L_total, L_data, L_phys, L_ic, L_cons

# ============================================================================
# Evaluation
# ============================================================================

def evaluate(model, t, S_t, E_t, I_t, R_t):
    model.eval()
    with torch.no_grad():
        pred = model(t)
    Sp = pred[:,0:1].cpu().numpy()
    Ep = pred[:,1:2].cpu().numpy()
    Ip = pred[:,2:3].cpu().numpy()
    Rp = pred[:,3:4].cpu().numpy()
    m  = {}
    for name, p, a in [('S',Sp,S_t.cpu().numpy()), ('E',Ep,E_t.cpu().numpy()),
                        ('I',Ip,I_t.cpu().numpy()), ('R',Rp,R_t.cpu().numpy())]:
        m[f'{name}_R2']   = r2_score(a, p)
        m[f'{name}_RMSE'] = np.sqrt(mean_squared_error(a, p))
        m[f'{name}_MAE']  = mean_absolute_error(a, p)
    return m, (Sp, Ep, Ip, Rp)

# ============================================================================
# FORWARD PROBLEM TRAINING
# ============================================================================

print("\n" + "="*60)
print("FORWARD PROBLEM: Prediction with known parameters")
print(f"  beta={config.BETA_TRUE}  sigma={config.SIGMA_TRUE}  gamma={config.GAMMA_TRUE}")
print("="*60)

model_fwd = PINN(config.LAYERS).to(device)
opt_fwd   = torch.optim.Adam(model_fwd.parameters(), lr=config.LEARNING_RATE)
hist_fwd  = {'total':[], 'data':[], 'physics':[], 'ic':[], 'conservation':[], 'test_total':[]}

for epoch in range(1, config.EPOCHS_FORWARD + 1):
    model_fwd.train()
    L, Ld, Lp, Li, Lc = total_loss_fn(
        t_train_grad, model_fwd,
        S_train, E_train, I_train, R_train,
        config.BETA_TRUE, config.SIGMA_TRUE, config.GAMMA_TRUE)
    opt_fwd.zero_grad()
    L.backward()
    opt_fwd.step()
    hist_fwd['total'].append(L.item())
    hist_fwd['data'].append(Ld.item())
    hist_fwd['physics'].append(Lp.item())
    hist_fwd['ic'].append(Li.item())
    hist_fwd['conservation'].append(Lc.item())
    with torch.no_grad():
        model_fwd.eval()
        pred_te = model_fwd(t_test)
        Sp = pred_te[:,0:1]; Ep = pred_te[:,1:2]
        Ip = pred_te[:,2:3]; Rp = pred_te[:,3:4]
        L_te = torch.mean((Sp-S_test)**2 + (Ep-E_test)**2 + (Ip-I_test)**2 + (Rp-R_test)**2)
        hist_fwd['test_total'].append(L_te.item())
    if epoch % config.PRINT_EVERY == 0:
        print(f"  Epoch {epoch:5d}/{config.EPOCHS_FORWARD} | "
              f"Train: {L.item():.6f} | "
              f"Test: {L_te.item():.6f} | "
              f"Data: {Ld.item():.6f} | "
              f"Physics: {Lp.item():.6f}")

m_tr_fwd, p_tr_fwd = evaluate(model_fwd, t_train, S_train, E_train, I_train, R_train)
m_te_fwd, p_te_fwd = evaluate(model_fwd, t_test,  S_test,  E_test,  I_test,  R_test)

print(f"\n  Training - S: {m_tr_fwd['S_R2']:.3f}  I: {m_tr_fwd['I_R2']:.3f}  R: {m_tr_fwd['R_R2']:.3f}")
print(f"  Test     - S: {m_te_fwd['S_R2']:.3f}  I: {m_te_fwd['I_R2']:.3f}  R: {m_te_fwd['R_R2']:.3f}")

# ============================================================================
# INVERSE PROBLEM TRAINING
# ============================================================================

print("\n" + "="*60)
print("INVERSE PROBLEM: Parameter estimation from data")
print(f"  Initial guesses: beta=0.3  sigma=0.2  gamma=0.1")
print("="*60)

model_inv     = PINN(config.LAYERS).to(device)
beta_learned  = nn.Parameter(torch.tensor([0.3], device=device))
sigma_learned = nn.Parameter(torch.tensor([0.2], device=device))
gamma_learned = nn.Parameter(torch.tensor([0.1], device=device))

opt_inv    = torch.optim.Adam(
    list(model_inv.parameters()) + [beta_learned, sigma_learned, gamma_learned],
    lr=config.LEARNING_RATE)
hist_inv   = {'total':[], 'data':[], 'physics':[], 'ic':[], 'conservation':[], 'test_total':[]}
param_hist = {'beta':[], 'sigma':[], 'gamma':[]}

for epoch in range(1, config.EPOCHS_INVERSE + 1):
    model_inv.train()
    L, Ld, Lp, Li, Lc = total_loss_fn(
        t_train_grad, model_inv,
        S_train, E_train, I_train, R_train,
        beta_learned, sigma_learned, gamma_learned)
    opt_inv.zero_grad()
    L.backward()
    opt_inv.step()
    with torch.no_grad():
        beta_learned.clamp_(0.01, 2.0)
        sigma_learned.clamp_(0.01, 1.0)
        gamma_learned.clamp_(0.01, 0.5)
    hist_inv['total'].append(L.item())
    hist_inv['data'].append(Ld.item())
    hist_inv['physics'].append(Lp.item())
    hist_inv['ic'].append(Li.item())
    hist_inv['conservation'].append(Lc.item())
    param_hist['beta'].append(beta_learned.item())
    param_hist['sigma'].append(sigma_learned.item())
    param_hist['gamma'].append(gamma_learned.item())
    with torch.no_grad():
        model_inv.eval()
        pred_te = model_inv(t_test)
        Sp = pred_te[:,0:1]; Ep = pred_te[:,1:2]
        Ip = pred_te[:,2:3]; Rp = pred_te[:,3:4]
        L_te_inv = torch.mean((Sp-S_test)**2 + (Ep-E_test)**2 + (Ip-I_test)**2 + (Rp-R_test)**2)
        hist_inv['test_total'].append(L_te_inv.item())
    if epoch % config.PRINT_EVERY == 0:
        print(f"  Epoch {epoch:5d}/{config.EPOCHS_INVERSE} | "
              f"Train: {L.item():.6f} | "
              f"Test: {L_te_inv.item():.6f} | "
              f"beta: {beta_learned.item():.4f} | "
              f"sigma: {sigma_learned.item():.4f} | "
              f"gamma: {gamma_learned.item():.4f}")

beta_est  = beta_learned.item()
sigma_est = sigma_learned.item()
gamma_est = gamma_learned.item()

m_tr_inv, p_tr_inv = evaluate(model_inv, t_train, S_train, E_train, I_train, R_train)
m_te_inv, p_te_inv = evaluate(model_inv, t_test,  S_test,  E_test,  I_test,  R_test)

print(f"\n  beta  : True={config.BETA_TRUE:.4f}  Estimated={beta_est:.4f}  Error={abs(config.BETA_TRUE-beta_est)/config.BETA_TRUE*100:.2f}%")
print(f"  sigma : True={config.SIGMA_TRUE:.4f}  Estimated={sigma_est:.4f}  Error={abs(config.SIGMA_TRUE-sigma_est)/config.SIGMA_TRUE*100:.2f}%")
print(f"  gamma : True={config.GAMMA_TRUE:.4f}  Estimated={gamma_est:.4f}  Error={abs(config.GAMMA_TRUE-gamma_est)/config.GAMMA_TRUE*100:.2f}%")
print(f"  R0    : True={config.BETA_TRUE/config.GAMMA_TRUE:.2f}  Estimated={beta_est/gamma_est:.2f}")
print(f"\n  Training - S: {m_tr_inv['S_R2']:.3f}  I: {m_tr_inv['I_R2']:.3f}  R: {m_tr_inv['R_R2']:.3f}")
print(f"  Test     - S: {m_te_inv['S_R2']:.3f}  I: {m_te_inv['I_R2']:.3f}  R: {m_te_inv['R_R2']:.3f}")

# ============================================================================
# Numpy arrays for plotting
# ============================================================================

t_np    = t.detach().cpu().numpy()
t_tr_np = t_train.detach().cpu().numpy()
t_te_np = t_test.detach().cpu().numpy()
S_np    = S.detach().cpu().numpy()
I_np    = I.detach().cpu().numpy()
R_np    = R.detach().cpu().numpy()
ep_arr  = np.arange(1, config.EPOCHS_FORWARD + 1)
ep_inv  = np.arange(1, config.EPOCHS_INVERSE + 1)
split_day = t_tr_np[-1][0]

# ============================================================================
# Figure 1: Forward Predictions
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Forward Problem Predictions - Synthetic Data', fontsize=14, fontweight='bold')

pairs = [
    (axes[0,0], S_np, p_tr_fwd[0], p_te_fwd[0], 'Susceptible (S)'),
    (axes[0,1], I_np, p_tr_fwd[2], p_te_fwd[2], 'Infected (I)'),
    (axes[1,0], R_np, p_tr_fwd[3], p_te_fwd[3], 'Recovered (R)'),
]
for ax, data, ptr, pte, title in pairs:
    ax.plot(t_np,    data*100, 'o',  color='steelblue',  markersize=3, alpha=0.5, label='Observed Data')
    ax.plot(t_tr_np, ptr*100,  '-',  color='darkorange',  linewidth=2,  label='PINN (train)')
    ax.plot(t_te_np, pte*100,  '--', color='green',        linewidth=2,  label='PINN (test)')
    ax.axvline(x=split_day, color='gray', linestyle=':', linewidth=1.2)
    ax.set_xlabel('Days')
    ax.set_ylabel('Population (%)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[1,1].plot(t_tr_np, p_tr_fwd[0]*100, color='steelblue',  linewidth=2, label='S')
axes[1,1].plot(t_tr_np, p_tr_fwd[2]*100, color='darkorange',  linewidth=2, label='I')
axes[1,1].plot(t_tr_np, p_tr_fwd[3]*100, color='green',        linewidth=2, label='R')
axes[1,1].set_xlabel('Days')
axes[1,1].set_ylabel('Population (%)')
axes[1,1].set_title('All Compartments (Training)')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_forward_predictions.png', dpi=300, bbox_inches='tight')
print("\nSaved: synthetic_forward_predictions.png")
plt.show()

# ============================================================================
# Figure 2: Inverse Predictions
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Inverse Problem Predictions - Synthetic Data', fontsize=14, fontweight='bold')

pairs = [
    (axes[0,0], S_np, p_tr_inv[0], p_te_inv[0], 'Susceptible (S)'),
    (axes[0,1], I_np, p_tr_inv[2], p_te_inv[2], 'Infected (I)'),
    (axes[1,0], R_np, p_tr_inv[3], p_te_inv[3], 'Recovered (R)'),
]
for ax, data, ptr, pte, title in pairs:
    ax.plot(t_np,    data*100, 'o',  color='steelblue',  markersize=3, alpha=0.5, label='Observed Data')
    ax.plot(t_tr_np, ptr*100,  '-',  color='darkorange',  linewidth=2,  label='PINN (train)')
    ax.plot(t_te_np, pte*100,  '--', color='green',        linewidth=2,  label='PINN (test)')
    ax.axvline(x=split_day, color='gray', linestyle=':', linewidth=1.2)
    ax.set_xlabel('Days')
    ax.set_ylabel('Population (%)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[1,1].plot(t_tr_np, p_tr_inv[0]*100, color='steelblue',  linewidth=2, label='S')
axes[1,1].plot(t_tr_np, p_tr_inv[2]*100, color='darkorange',  linewidth=2, label='I')
axes[1,1].plot(t_tr_np, p_tr_inv[3]*100, color='green',        linewidth=2, label='R')
axes[1,1].set_xlabel('Days')
axes[1,1].set_ylabel('Population (%)')
axes[1,1].set_title('All Compartments (Training)')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_inverse_predictions.png', dpi=300, bbox_inches='tight')
print("Saved: synthetic_inverse_predictions.png")
plt.show()

# ============================================================================
# Figure 3: Combined Training vs Test Loss
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Training vs Test Loss - Synthetic Data', fontsize=13, fontweight='bold')

axes[0].semilogy(ep_arr, hist_fwd['total'],      color='steelblue',  linewidth=2, label='Training Loss')
axes[0].semilogy(ep_arr, hist_fwd['test_total'], color='crimson',     linewidth=2, label='Testing Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Forward Problem')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(ep_inv, hist_inv['total'],      color='steelblue',  linewidth=2, label='Training Loss')
axes[1].semilogy(ep_inv, hist_inv['test_total'], color='crimson',     linewidth=2, label='Testing Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Inverse Problem')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_train_test_loss.png', dpi=300, bbox_inches='tight')
print("Saved: synthetic_train_test_loss.png")
plt.show()

# ============================================================================
# Figure 4: Forward Loss Curves
# ============================================================================

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Forward Problem - Loss Curves (Synthetic)', fontsize=13, fontweight='bold')

milestones = [3000, 6000, 9000, 12000, 15000]
loss_panels = [
    ('total',        'navy',        'Total Loss'),
    ('data',         'darkorange',  'Data Loss'),
    ('physics',      'green',       'Physics Loss'),
    ('conservation', 'purple',      'Conservation Loss'),
]
for ax, (key, color, title) in zip(axes, loss_panels):
    ax.semilogy(ep_arr, hist_fwd[key], color=color, linewidth=1.5)
    for ms in milestones:
        ax.axvline(x=ms, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_forward_loss.png', dpi=300, bbox_inches='tight')
print("Saved: synthetic_forward_loss.png")
plt.show()

# ============================================================================
# Figure 5: Inverse Loss Curves
# ============================================================================

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Inverse Problem - Loss Curves (Synthetic)', fontsize=13, fontweight='bold')

for ax, (key, color, title) in zip(axes, loss_panels):
    ax.semilogy(ep_inv, hist_inv[key], color=color, linewidth=1.5)
    for ms in milestones:
        ax.axvline(x=ms, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_inverse_loss.png', dpi=300, bbox_inches='tight')
print("Saved: synthetic_inverse_loss.png")
plt.show()

# ============================================================================
# Figure 6: Parameter Convergence
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Parameter Convergence - Inverse Problem (Synthetic)', fontsize=13, fontweight='bold')

param_panels = [
    ('beta',  config.BETA_TRUE,  'steelblue',   'Infection Rate (beta)'),
    ('sigma', config.SIGMA_TRUE, 'darkorange',  'Incubation Rate (sigma)'),
    ('gamma', config.GAMMA_TRUE, 'green',       'Recovery Rate (gamma)'),
]
for ax, (key, true_val, color, title) in zip(axes, param_panels):
    ax.plot(ep_inv, param_hist[key], color=color, linewidth=1.5, label='Estimated')
    ax.axhline(y=true_val, color='red', linestyle='--', linewidth=2,
               label=f'True value ({true_val:.3f})')
    for ms in milestones:
        ax.axvline(x=ms, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(key)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_parameters.png', dpi=300, bbox_inches='tight')
print("Saved: synthetic_parameters.png")
plt.show()

# ============================================================================
# Save results
# ============================================================================

results = {
    'dataset'       : 'synthetic',
    'forward_train' : m_tr_fwd,
    'forward_test'  : m_te_fwd,
    'inverse_train' : m_tr_inv,
    'inverse_test'  : m_te_inv,
    'parameters'    : {
        'beta_true'       : config.BETA_TRUE,
        'sigma_true'      : config.SIGMA_TRUE,
        'gamma_true'      : config.GAMMA_TRUE,
        'beta_estimated'  : beta_est,
        'sigma_estimated' : sigma_est,
        'gamma_estimated' : gamma_est,
    }
}
torch.save(results, 'synthetic_results.pth')
print("Saved: synthetic_results.pth")

print("\n" + "="*60)
print("SYNTHETIC COMPLETE - 6 figures saved")
print("="*60)